In [0]:
import json
import urllib.request
from abc import ABC, abstractmethod
from typing import Dict, Any
from pyspark.sql import SparkSession, DataFrame

In [0]:
class DataFetcher(ABC):
    """Interface for fetching external data."""
    @abstractmethod
    def fetch_json(self, url: str) -> Dict[str, Any]:
        pass

class StorageManager(ABC):
    """Interface for writing PySpark DataFrames to storage."""
    @abstractmethod
    def save_as_delta(self, df: DataFrame, table_name: str, path: str) -> None:
        pass

In [0]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

class CMSApiClient(DataFetcher):
    """Robust CMS API client with automated retries and exponential backoff."""
    
    def __init__(self, retries: int = 3, backoff_factor: float = 1.0, timeout: int = 45):
        self.timeout = timeout
        self.session = requests.Session()
        
        # Configure retry strategy for transient network errors & HTTP 5xx responses
        retry_strategy = Retry(
            total=retries,
            backoff_factor=backoff_factor,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET"]
        )
        
        adapter = HTTPAdapter(max_retries=retry_strategy)
        self.session.mount("https://", adapter)
        self.session.mount("http://", adapter)
        
        # Set realistic browser headers to prevent API drops
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'application/json, text/plain, */*'
        })

    def fetch_json(self, url: str) -> Dict[str, Any]:
        try:
            response = self.session.get(url, timeout=self.timeout, verify=True)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.Timeout:
            raise RuntimeError(
                f"Connection timed out after {self.timeout}s while reaching {url}. "
                "Ensure your Databricks cluster has outbound internet access enabled."
            )
        except requests.exceptions.RequestException as e:
            raise RuntimeError(f"API Request failed: {str(e)}")

In [0]:
class SparkStorageManager(StorageManager):
    """Handles PySpark Delta table writes optimized for Databricks Free/Community edition."""
    
    def save_as_delta(self, df: DataFrame, table_name: str, path: str) -> None:
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .option("path", path)
            .saveAsTable(table_name)
        )
        print(f"Successfully saved {df.count()} records to Delta Table: '{table_name}'")

In [0]:
class CoreSetReader:
    """Parses raw payload data into strongly typed PySpark DataFrames."""
    
    def __init__(self, spark: SparkSession):
        self.spark = spark

    def raw_to_dataframe(self, json_data: Dict[str, Any]) -> DataFrame:
        """Converts raw JSON payload into a distributed PySpark DataFrame."""
        # Extract data payload list if encapsulated under JSON API standards
        rows = json_data.get("data", json_data) if isinstance(json_data, dict) else json_data
        
        # Convert dictionary list to PySpark RDD then DataFrame
        rdd = self.spark.sparkContext.parallelize([json.dumps(row) for row in rows])
        df = self.spark.read.json(rdd)
        return df

In [0]:
class CoreSetPipeline:
    """Facade orchestrating the download, parsing, and storage execution pipeline."""
    
    # Public CMS Data API endpoints for Child and Adult Core Set measure datasets
    ENDPOINTS = {
        "child_core_set": "https://data.cms.gov/data-api/v1/dataset/a62c5c06-1896-410a-a53d-24ecf23ee6d0/data?size=5000",
        "adult_core_set": "https://data.cms.gov/data-api/v1/dataset/8676d5e1-88f6-49a0-97c2-ec061c4d924d/data?size=5000"
    }

    def __init__(
        self, 
        spark: SparkSession, 
        fetcher: DataFetcher, 
        storage: StorageManager,
        base_dbfs_path: str = "dbfs:/user/hive/warehouse/cms_core_sets"
    ):
        self.spark = spark
        self.fetcher = fetcher
        self.storage = storage
        self.reader = CoreSetReader(spark)
        self.base_dbfs_path = base_dbfs_path

    def run(self) -> None:
        """Executes the extraction and ingestion steps for both Core Sets."""
        for dataset_key, url in self.ENDPOINTS.items():
            print(f"\n--- Processing CMS Dataset: {dataset_key} ---")
            
            # Step 1: Extract
            raw_json = self.fetcher.fetch_json(url)
            
            # Step 2: Transform to PySpark DataFrame
            df = self.reader.raw_to_dataframe(raw_json)
            
            # Step 3: Load into Databricks Delta Lake
            table_name = f"cms_{dataset_key}"
            target_path = f"{self.base_dbfs_path}/{dataset_key}"
            self.storage.save_as_delta(df, table_name, target_path)

In [0]:
# Initialize Dependency Injected objects
fetcher_service = CMSApiClient()
storage_service = SparkStorageManager()

# Execute Orchestrator Pipeline
pipeline = CoreSetPipeline(
    spark=spark,  # Native global spark session in Databricks
    fetcher=fetcher_service,
    storage=storage_service
)

In [0]:
pipeline.run()